In [5]:
# Run this first in Jupyter Notebook
!mamba install pandas

mambajs 0.19.13

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas
Channels: emscripten-forge, conda-forge

Solving environment...
Solving took 0.8432999999523163 seconds
  Name                          Version                       Build                         Channel                       
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
+ pandas                        3.0.2                         np22py313h9d9dc1e_0           emscripten-forge              
+ python-tzdata                 2026.2                        pyhd8ed1ab_0                  conda-forge                   
- pip                           25.3                          pyh145f28c_0                  conda-forge                   


In [6]:
!mamba install ipywidgets

mambajs 0.19.13

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas, ipywidgets
Channels: emscripten-forge, conda-forge

Solving environment...
Solving took 0.9676000000238418 seconds
All requested packages already installed.


In [13]:
# ======================================================
# Monthly Budget Tracker
# Save Output in CSV + Table Format (Overwrite Mode)
# Jupyter Notebook Compatible
# ======================================================

import pandas as pd
from IPython.display import display, HTML, FileLink
import ipywidgets as widgets
from datetime import date

# ======================================================
# CSV FILE
# ======================================================

CSV_FILE = "monthly_budget.csv"

# ======================================================
# Empty DataFrame
# ======================================================

df = pd.DataFrame(columns=[
    "Category",
    "Budget",
    "Spent",
    "Payment Method",
    "Transaction Date"
])

# ======================================================
# Currency Formatter
# ======================================================

def format_inr(x):
    return f"₹{x:,.0f}"

# ======================================================
# Generate Table HTML
# ======================================================

def generate_table():

    temp_df = df.copy()

    if not temp_df.empty:

        temp_df["Remaining"] = (
            temp_df["Budget"].astype(int)
            - temp_df["Spent"].astype(int)
        )

        display_df = temp_df.copy()

        display_df["Budget"] = display_df["Budget"].apply(format_inr)
        display_df["Spent"] = display_df["Spent"].apply(format_inr)
        display_df["Remaining"] = display_df["Remaining"].apply(format_inr)

    else:
        display_df = temp_df

    styles = """
    <style>

    .title{
        background:linear-gradient(90deg,#ff512f,#dd2476);
        color:white;
        padding:25px;
        font-size:50px;
        font-weight:bold;
        border-radius:15px 15px 0px 0px;
        font-family:Arial;
    }

    table{
        width:100%;
        border-collapse:collapse;
        font-family:Arial;
        background:white;
        box-shadow:0px 4px 10px rgba(0,0,0,0.2);
    }

    th{
        background:#f7df74;
        padding:14px;
        text-align:center;
        font-size:18px;
    }

    td{
        padding:12px;
        text-align:center;
        border-bottom:1px solid #ddd;
    }

    tr:nth-child(even){
        background:#f9f9f9;
    }

    tr:hover{
        background:#fff4dd;
    }

    </style>
    """

    title = """
    <div class='title'>
    Monthly Budget
    </div>
    """

    return styles + title + display_df.to_html(index=False)

# ======================================================
# SAVE CSV (OVERWRITE MODE)
# ======================================================

def save_csv():

    save_df = df.copy()

    if not save_df.empty:

        save_df["Remaining"] = (
            save_df["Budget"].astype(int)
            - save_df["Spent"].astype(int)
        )

    # OVERWRITE EXISTING CSV
    save_df.to_csv(CSV_FILE, index=False)

# ======================================================
# Widgets
# ======================================================

category_dropdown = widgets.Dropdown(
    options=[
        "🏠 Rent",
        "🛒 Grocery",
        "💰 Savings",
        "💡 Electricity",
        "📶 Internet",
        "🍔 Food",
        "🚕 Transport",
        "🎬 Entertainment"
    ],
    description='Category:',
    layout=widgets.Layout(width='350px')
)

budget_input = widgets.IntText(
    description='Budget:',
    value=0,
    layout=widgets.Layout(width='350px')
)

spent_input = widgets.IntText(
    description='Spent:',
    value=0,
    layout=widgets.Layout(width='350px')
)

payment_dropdown = widgets.Dropdown(
    options=[
        "UPI SAV",
        "UPI CC",
        "BANK SAV",
        "CC"
    ],
    description='Payment:',
    layout=widgets.Layout(width='350px')
)

date_picker = widgets.DatePicker(
    description='Date:',
    value=date.today(),
    layout=widgets.Layout(width='350px')
)

add_button = widgets.Button(
    description='Add Entry',
    button_style='success',
    layout=widgets.Layout(width='200px', height='45px')
)

download_button = widgets.Button(
    description='Download CSV',
    button_style='info',
    layout=widgets.Layout(width='200px', height='45px')
)

table_output = widgets.HTML()
download_output = widgets.Output()

# ======================================================
# Add Entry Function
# ======================================================

def add_entry(b):

    global df

    new_row = pd.DataFrame([{
        "Category": category_dropdown.value,
        "Budget": budget_input.value,
        "Spent": spent_input.value,
        "Payment Method": payment_dropdown.value,
        "Transaction Date": str(date_picker.value)
    }])

    # Add New Row
    df = pd.concat([df, new_row], ignore_index=True)

    # OVERWRITE CSV FILE
    save_csv()

    # Update Table
    table_output.value = generate_table()

    # Reset Inputs
    budget_input.value = 0
    spent_input.value = 0

# ======================================================
# Download CSV
# ======================================================

def download_csv(b):

    save_csv()

    download_output.clear_output()

    with download_output:
        display(FileLink(CSV_FILE))

# ======================================================
# Button Actions
# ======================================================

add_button.on_click(add_entry)
download_button.on_click(download_csv)

# ======================================================
# Initial Table
# ======================================================

table_output.value = generate_table()

# ======================================================
# Final Layout
# ======================================================

ui = widgets.VBox([

    category_dropdown,
    budget_input,
    spent_input,
    payment_dropdown,
    date_picker,

    widgets.HBox([
        add_button,
        download_button
    ]),

    table_output,
    download_output

])

display(ui)